# PCAF MathArena Competition Result Analysis

This notebook processes result logs from various mathematics competitions (AIME, HMMT, SMT, etc.) to evaluate model performance. It aggregates individual CSV files to calculate accuracy, efficiency, and self-correction capabilities.

In [ ]:
import pandas as pd
import glob
import ast
import os

### Metrics Calculation

The function below, `calculate_competition_metrics`, iterates through the result files to compute three key metrics:

* **Accuracy:** The overall success rate (`final_score`) for the competition.
* **Avg Turns:** The average number of interaction turns required. Lower values indicate the model reached a conclusion more efficiently.
* **Recovery Rate:** A measure of resilience. This tracks how often the model successfully arrived at the correct answer *after* failing the first turn (i.e., self-correction).

In [ ]:
def calculate_competition_metrics(file_pattern="final_results/PCAF_*.csv"):
    files = glob.glob(file_pattern)
    results = []

    for file in files:
        df = pd.read_csv(file)
        
        # 1. Calculate Accuracy (Mean of final_score)
        accuracy = df['final_score'].mean()
        
        # 2. Parse lists for advanced metrics (Recovery Rate & Avg Turns)
        def safe_eval(val):
            try: return ast.literal_eval(val)
            except: return []

        scores_list = df['pcaf_scores'].apply(safe_eval)
        turns_list = df['pcaf_turns'].apply(safe_eval)
        
        initial_fails = 0
        recovered_runs = 0
        all_turns = []
        
        for row_scores, row_turns in zip(scores_list, turns_list):
            for score, turn in zip(row_scores, row_turns):
                all_turns.append(turn)
                # A "Recovery" is a successful attempt (score=1) that failed on Turn 1 (turn > 1)
                if turn > 1:
                    initial_fails += 1
                    if score == 1:
                        recovered_runs += 1
        
        recovery_rate = recovered_runs / initial_fails if initial_fails > 0 else 0
        avg_turns = sum(all_turns) / len(all_turns) if all_turns else 0
        
        # Extract name and clean up
        comp_name = os.path.basename(file).replace("PCAF_Final_Result_", "").replace(".csv", "").upper()
        
        results.append({
            "Competition": comp_name,
            "Accuracy": f"{accuracy:.2%}",
            "Avg Turns": round(avg_turns, 2),
            "Recovery Rate": f"{recovery_rate:.1%}"
        })

    return pd.DataFrame(results).sort_values("Competition")

### Summary of Results

The table below displays the aggregated statistics for each competition dataset found in the `final_results/` directory, sorted by competition name.

In [7]:
# Run and display
results_df = calculate_competition_metrics()
print(results_df.to_string(index=False))

  Competition Accuracy  Avg Turns Recovery Rate
    AIME_2025   56.67%       2.16         29.5%
   BRUMO_2025   65.83%       1.91         48.2%
        CMIMC   51.25%       2.56         41.6%
HMMT_FEB_2025   45.83%       2.47         35.0%
HMMT_NOV_2025   46.67%       2.28         25.4%
     SMT_2025   55.66%       2.03         36.7%
